In [1]:
# Defining all imports here
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-31 12:38:15--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.005s  

2026-03-31 12:38:15 (198 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
with open('input.txt', mode='r', encoding='utf-8') as f:
  text = f.read()

In [ ]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [ ]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
# Getting vocab size
chars = list(sorted(set(text)))
itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
vocab_size=len(chars)
print(vocab_size)
"".join(chars)

65


"\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"

In [ ]:
# defining encode and decode functions
encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

print(encode('hi there'))
print(decode(encode('hi there')))

[46, 47, 1, 58, 46, 43, 56, 43]
['h', 'i', ' ', 't', 'h', 'e', 'r', 'e']


In [ ]:
# Similar to tokenizer like tiktoken but
# just working on a single char here instead of a sub word and much smaller vocab size

data = torch.tensor(encode(text), dtype=torch.int64)
data.shape, data.dtype

(torch.Size([1115394]), torch.int64)

In [ ]:
data[:10]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])

In [ ]:
# Defining block size i.e. context length
# and defining procedure to build X and Y
block_size = 8

print(data[:block_size+1])
# A toy example
X = data[0:block_size]
Y = data[1:block_size+1]

for t in range(8):
  x = X[:t+1] #include until current char OR time dimension
  y = Y[t]
  print(f"Input->Target; {x} -> {y}")


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])
Input->Target; tensor([18]) -> 47
Input->Target; tensor([18, 47]) -> 56
Input->Target; tensor([18, 47, 56]) -> 57
Input->Target; tensor([18, 47, 56, 57]) -> 58
Input->Target; tensor([18, 47, 56, 57, 58]) -> 1
Input->Target; tensor([18, 47, 56, 57, 58,  1]) -> 15
Input->Target; tensor([18, 47, 56, 57, 58,  1, 15]) -> 47
Input->Target; tensor([18, 47, 56, 57, 58,  1, 15, 47]) -> 58


In [ ]:
# Defining data split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
train_data.shape, val_data.shape

(torch.Size([1003854]), torch.Size([111540]))

In [ ]:
# Defining batch loader
# So from example above
# One batch would be X: [18, 47, 56, 57, 58, 1, 15, 47]   Y: [47, 56, 57, 58, 1, 15, 47, 58]
# That means somewhere in my network, the way I generated the examples above should be accounted


def get_batch(split='train', batch_size=32):
  if split == 'train':
    data = train_data
  elif split == 'val':
    data = val_data

  ix = torch.randint(0, data.shape[0]-block_size, (batch_size,))
  #print(ix.shape, ix, data[ix[0]]) # [32]
  list_X = [data[i: i+block_size] for i in ix]
  X = torch.stack(list_X, dim=0) # [32, 8]
  #print(X.shape, X[:2])
  Y = torch.stack([data[i+1: i+block_size+1] for i in ix], dim=0) # [32, 8]
  #print(Y.shape, Y[:2])
  return X, Y


get_batch()

(tensor([[ 1, 47, 52,  1, 34, 47, 43, 52],
         [46, 43,  1, 60, 47, 53, 50, 43],
         [32, 46, 43, 52,  1, 51, 59, 57],
         [46, 43, 47, 56,  1, 41, 53, 59],
         [50, 39, 63, 43, 56, 10,  0, 21],
         [56,  1, 41, 53, 52, 57, 41, 47],
         [57, 46, 53, 59, 50, 42,  1, 46],
         [43, 12,  0,  0, 29, 33, 17, 17],
         [39, 56, 43,  1, 61, 43, 50, 50],
         [26, 19, 20, 13, 25, 10,  0, 33],
         [39, 52, 58,  1, 44, 53, 56,  1],
         [56, 53, 41, 43, 43, 42,  1, 61],
         [43,  1, 39, 40, 57, 43, 52, 58],
         [32, 13, 10,  0, 13, 63,  6,  1],
         [ 1, 21,  1, 44, 43, 39, 56,  6],
         [46, 47, 57,  1, 51, 39, 52,  6],
         [59, 42,  1, 55, 59, 43, 43, 52],
         [39, 42, 43, 42,  6,  1, 46, 53],
         [ 1, 58, 53,  1, 51, 43,  1, 61],
         [39, 56, 42, 57,  0, 20, 47, 57],
         [58, 50, 63,  6,  1, 54, 53, 47],
         [ 1, 58, 56, 47, 40, 59, 52, 43],
         [61, 47, 50, 50, 11,  1, 39, 52],
         [3

In [ ]:
# Defining Bigram model

class BigramModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=vocab_size)

  def forward(self, input, targets = None):
    # x dim are (B,T) (batch, time)
    logits = self.embedding_table(input) # (B, T, C) -> (batch, time, channels) -> channels is embedding size here
    #print(f"Logits: {logits.shape}, Targets: {targets.shape}")

    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      #print(logits[1,0]) # La
      logits = logits.view(B*T, C)
      #print(logits[block_size]) # Lb ; La=Lb
      targets = targets.view(B*T)
      #print(f"After Logits: {logits.shape}, Targets: {targets.shape}")
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, input, num_tokens=200):

    out = []
    for _ in range(num_tokens):
      logits, _ = self(input) # [1, T, 65] # Initial [1,1,65]
      # print(logits.shape)
      # X shape [1, T] ; Logits shape [1, T, C]
      # I'm only interested in last index of T of logits as that's the next prediction
      last_T = logits[:, -1, :] # [1, 65]
      # print(last_T.shape) #[1, 65]
      probs = F.softmax(last_T, dim=1)
      # print(probs.shape) [1, 65]
      next_idx = torch.multinomial(probs, num_samples=1) # [1,1]
      #print(next_idx.shape) # [1, 1]
      out.append(next_idx.item())
      # print(input[:, 1:].shape) # [1, T-1]
      #input = torch.cat((input[:, 1:], next_idx), dim=1)
      input = torch.cat((input, next_idx), dim=1) # [1, T+1]
      #print(input.shape)

    print(f"=========== Generated output ==========\n")
    print("".join([itos[i] for i in out]))



model = BigramModel(vocab_size)
Xb, Yb = get_batch()
#print(f"Xb: {Xb.shape} Yb: {Yb.shape}") # Xb: [32, 8] Yb: [32, 8]
logits, loss = model(Xb, Yb)
#print(loss)


In [ ]:
# Data generation before training
start_phrase = torch.zeros((1, block_size), dtype=torch.long)
model.generate(start_phrase)

=========== Generated output ==========

B?UPDknjVzxLq3XdQ'xupZCFgEMeKO 'JaL,qlRdZXCfrCofFXeAZE
VqL-C?OvfredjzZ:D3, U
H;wAwHB.
zxuuUf.BHe?M-$quqcx gPfkirD&;H!O3hUOvw-3hZrUSp:aJZZsDCIiezCm jhN-XddaLxLkJsMOXQoFMeh'dX;L-iOU&us,,ockY$ :cAk!UcMS;


In [ ]:
# Expected starting loss
import math
-(math.log(1/vocab_size))

4.174387269895637

In [ ]:
@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train', 'val']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']} Eval loss {losses['val']}")

In [ ]:
# Training the model
steps = 20_000
lr = 3e-4
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for i in range(steps):
  Xb, Yb = get_batch()
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  optimizer.step()

  if i % 500 == 0:
    evaluate_loss(eval_iters)
    #print(f"{i}/{steps} = {loss.item()}")


Train loss 4.595327854156494 Eval loss 4.588888645172119
Train loss 4.42292594909668 Eval loss 4.42202615737915
Train loss 4.256426811218262 Eval loss 4.261200428009033
Train loss 4.10536003112793 Eval loss 4.110866069793701
Train loss 3.9605133533477783 Eval loss 3.954420804977417
Train loss 3.819626808166504 Eval loss 3.8238463401794434
Train loss 3.7042200565338135 Eval loss 3.7005600929260254
Train loss 3.5736005306243896 Eval loss 3.587937831878662
Train loss 3.4677534103393555 Eval loss 3.4758758544921875
Train loss 3.376096248626709 Eval loss 3.370819330215454
Train loss 3.2841107845306396 Eval loss 3.2806177139282227
Train loss 3.1964871883392334 Eval loss 3.195564031600952
Train loss 3.1214263439178467 Eval loss 3.129140615463257
Train loss 3.053375482559204 Eval loss 3.050929069519043
Train loss 2.981755256652832 Eval loss 2.9921514987945557
Train loss 2.92877197265625 Eval loss 2.938927412033081
Train loss 2.8845574855804443 Eval loss 2.8832578659057617
Train loss 2.83817958

In [ ]:
# Data generation post some training
start_phrase = torch.zeros((1, 1), dtype=torch.long)
model.generate(start_phrase, num_tokens=1_000)

=========== Generated output ==========

OWisst, icher,LO: isthenp.
ThelSowin houst u r idoulgu,
TRI faithe:Xvess s te d
ARzkse
TENRICo indes, hsuatowher malldst, t.

Wkag l'le h.
Fi:
Alond, des.
TRLUplourlad men mys k, to mel RBatQghethe; an athe arit y in, m,
and ca, we n n lll h u cked wsove whot bend EOf winortee,
Shannu t tren TOFr nk, f nd my:
r
w s'dantqy?
No!'s, thefoHJere IZOunsam id th die br Peleowhoblshe y usins tousenglsthenoby ss r o! coCrd n alXEDoulifod beate os s3?Zobe t y ope:

Tirre!
FFZRSCO? vd wl IVIE:mee? lly ch Xbre!'lenge at he k;hilv!d, lt s shoris h fin.
My ifouss Go n, ayorlicathotht hid-cout ffermyomss.
w id mappseravesethagl ng, o? repplem;f calo he m.
Y:
An cerim edn,
BE:
Twseergisile oulotho ar,

ngere&Mofedilonveatooo; gt nd hes t h by bed! prt hit,
Pr br P;V:EWhand O s CINomebyrale ir we bll:
K; l'l'Watheet-verw! ises ncoovifourt
Coou't ll-w ve ar:

lougeryotenortas, thifow llix, y?
TIld it
TI
Fr we w renthe vepd E:
angurvppan ads, besertussvus.
I'juser

In [ ]:
# Only parameters is the embedding table
[p.shape for p in model.parameters()]

[torch.Size([65, 65])]

### Unified Code with GPU

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

with open('input.txt', mode='r', encoding='utf-8') as f:
  text = f.read()

chars = list(sorted(set(text)))
itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
vocab_size=len(chars)

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

data = torch.tensor(encode(text), dtype=torch.int64)

# Defining data split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"DEVICE is {device}")

block_size = 8

def get_batch(split='train', batch_size=32):
  if split == 'train':
    data = train_data
  elif split == 'val':
    data = val_data

  ix = torch.randint(0, data.shape[0]-block_size, (batch_size,))
  list_X = [data[i: i+block_size] for i in ix] # 32 list elements with each element of size 8
  X = torch.stack(list_X, dim=0) # [32, 8]
  Y = torch.stack([data[i+1: i+block_size+1] for i in ix], dim=0) # [32, 8]
  return X.to(device), Y.to(device)

@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train', 'val']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']} Eval loss {losses['val']}")

class BigramModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    logits = self.embedding_table(input) # (B, T, C) -> (batch, time, channels) -> channels is embedding size here
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, input, num_tokens=200):
    # Initial input [1, T]
    out = []
    for _ in range(num_tokens):
      logits, _ = self(input) # [1, T, 65]
      last_T = logits[:, -1, :] # [1, 65]
      probs = F.softmax(last_T, dim=1)
      next_idx = torch.multinomial(probs, num_samples=1) # [1, 1]
      out.append(next_idx.item())
      input = torch.cat((input, next_idx), dim=1) # [1, T]+[1, 1] -> [1, T+1]

    print(f"=========== Generated output ==========\n")
    print("".join([itos[i] for i in out]))



model = BigramModel(vocab_size)
model.to(device)
# Training the model
steps = 20_000
lr = 3e-4
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for i in range(steps):
  Xb, Yb = get_batch()
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()

  if i % 500 == 0:
    evaluate_loss(eval_iters)


# Generation
# Data generation post some training
start_phrase = torch.zeros((1, 1), dtype=torch.long, device=device)
model.generate(start_phrase, num_tokens=1_000)

DEVICE is cpu
Train loss 4.598545074462891 Eval loss 4.604114532470703
Train loss 4.428318977355957 Eval loss 4.4300923347473145
Train loss 4.257628440856934 Eval loss 4.266117095947266
Train loss 4.115076541900635 Eval loss 4.115722179412842
Train loss 3.9659204483032227 Eval loss 3.9827322959899902
Train loss 3.8370935916900635 Eval loss 3.8448596000671387
Train loss 3.706259250640869 Eval loss 3.726316452026367
Train loss 3.5986673831939697 Eval loss 3.612027645111084
Train loss 3.4839587211608887 Eval loss 3.500013828277588
Train loss 3.3884782791137695 Eval loss 3.4004175662994385
Train loss 3.2987353801727295 Eval loss 3.315098524093628
Train loss 3.2132773399353027 Eval loss 3.229717969894409
Train loss 3.1349213123321533 Eval loss 3.1490478515625
Train loss 3.0611636638641357 Eval loss 3.076413869857788
Train loss 3.001784086227417 Eval loss 3.0168068408966064
Train loss 2.935187339782715 Eval loss 2.962308645248413
Train loss 2.889261245727539 Eval loss 2.9017868041992188
Trai

#### Self-attention math tricks

In [ ]:
# Example for each T's embeddings being an average of last T embeddings
torch.manual_seed(1001)
xt = torch.randn((4,8,2))
print(xt[0])

# METHOD 1
xt2 = torch.zeros((4,8,2))
b, t, c = xt.shape
for bi in range(b):
  for ti in range(t):
    xt2[bi,ti] = xt[bi,:ti+1].mean(dim=0)
print(xt2[0])

# METHOD 2
wei = torch.tril(torch.ones(t,t))
# Normalizing
wei = wei/wei.sum(dim=1, keepdim=True)

xt3 = wei @ xt # [T, T] @ [B, T, C] -> [B, T, T] @ [B, T, C]
print(f"{xt3.shape} \n {xt3[0]}")

# METHOD 3
tril = torch.tril(torch.ones(t,t))
wei = torch.zeros((t,t)) # This is initial way of defining affinities (or similarity or pull of each Ti with any Tj)
wei = wei.masked_fill(tril == 0., -torch.inf) # Future can't communicate with the past
wei = F.softmax(wei, dim=1)
xt4 = wei @ xt
print(f"{xt4.shape} \n {xt4[0]}")

tensor([[ 0.8701,  0.1744],
        [ 1.4344, -0.5874],
        [-0.2707, -0.5549],
        [ 0.4075, -1.4089],
        [-0.4994,  0.5628],
        [ 1.3499,  0.1053],
        [ 0.5929, -0.6074],
        [ 0.5606,  0.3282]])
tensor([[ 0.8701,  0.1744],
        [ 1.1523, -0.2065],
        [ 0.6779, -0.3226],
        [ 0.6103, -0.5942],
        [ 0.3884, -0.3628],
        [ 0.5486, -0.2848],
        [ 0.5550, -0.3309],
        [ 0.5557, -0.2485]])
torch.Size([4, 8, 2]) 
 tensor([[ 0.8701,  0.1744],
        [ 1.1523, -0.2065],
        [ 0.6779, -0.3226],
        [ 0.6103, -0.5942],
        [ 0.3884, -0.3628],
        [ 0.5486, -0.2848],
        [ 0.5550, -0.3309],
        [ 0.5557, -0.2485]])
torch.Size([4, 8, 2]) 
 tensor([[ 0.8701,  0.1744],
        [ 1.1523, -0.2065],
        [ 0.6779, -0.3226],
        [ 0.6103, -0.5942],
        [ 0.3884, -0.3628],
        [ 0.5486, -0.2848],
        [ 0.5550, -0.3309],
        [ 0.5557, -0.2485]])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

with open('input.txt', mode='r', encoding='utf-8') as f:
  text = f.read()

chars = list(sorted(set(text)))
itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
vocab_size=len(chars)

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

data = torch.tensor(encode(text), dtype=torch.int64)

# Defining data split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


block_size = 8
n_embd = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"DEVICE is {device}")

def get_batch(split='train', batch_size=32):
  if split == 'train':
    data = train_data
  elif split == 'val':
    data = val_data

  ix = torch.randint(0, data.shape[0]-block_size, (batch_size,))
  list_X = [data[i: i+block_size] for i in ix] # 32 list elements with each element of size 8
  X = torch.stack(list_X, dim=0) # [32, 8]
  Y = torch.stack([data[i+1: i+block_size+1] for i in ix], dim=0) # [32, 8]
  return X.to(device), Y.to(device)

@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train', 'val']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']} Eval loss {losses['val']}")

class AttnBlock(nn.Module):
  def __init__(self):
    super().__init__()
    self.Q = nn.Linear(n_embd, n_embd, bias=False) # Stores [out_features, in_features] and X @ W.T
    self.K = nn.Linear(n_embd, n_embd, bias=False)
    self.V = nn.Linear(n_embd, n_embd, bias=False)
    #https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, input):
    # input [B, T, n_embd]
    B,T,C = input.shape
    query = self.Q(input) # [n_embd, n_embd] broadcasted to [B, n_embd, n_embd]; [B, T, n_embd] @ [B, n_embd, n_embd]
    key = self.K(input) # [B, T, n_embd]
    value = self.V(input) # [B, T, n_embd]
    # print(f"I: {input.shape} K: {key.shape} Q: {query.shape} KT: {torch.transpose(key, -2, -1).shape}")

    wei = query @ torch.transpose(key, -2, -1) # [B, T, n_embd] @ [B, n_embd, T] -> [B, T, T]
    wei = wei * (C**-0.5) # Scaling down
    # Until this wei is initial affinities based on data

    wei = wei.masked_fill(self.tril==0, -torch.inf)
    wei = F.softmax(wei, dim=-1)
    out = wei @ value # [B,T,T] @ [B,T,n_embd] -> [B,T,n_embd]
    return out

class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
    self.position_table = nn.Embedding(block_size, n_embd)
    self.sa_head = AttnBlock()
    self.lm_head = nn.Linear(in_features=n_embd, out_features=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    B, T = input.shape
    token_embd = self.embedding_table(input) # (B, T, embd) -> (batch, time, channels) -> channels is embedding size here
    pos_input = torch.arange(T).repeat(B,1) # [B, T] [[0,1,2...7] ... 8 batch]
    pos_embd = self.position_table(pos_input) # (B, T, embd)
    # Alternatively can do
    # pos_embd = self.position_table(torch.arange(T)) # [T, embd] (would work in next step because of broadcasting)
    x = token_embd + pos_embd
    x = self.sa_head(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, input, num_tokens=200):
    # Initial input [1, T]
    out = []
    for _ in range(num_tokens):
      logits, _ = self(input) # [1, T, 65]
      last_T = logits[:, -1, :] # [1, 65]
      probs = F.softmax(last_T, dim=1)
      next_idx = torch.multinomial(probs, num_samples=1) # [1, 1]
      out.append(next_idx.item())
      input = torch.cat((input[:,1:], next_idx), dim=1) # [1, T-1]+[1, 1] -> [1, T]

    print(f"=========== Generated output ==========\n")
    print("".join([itos[i] for i in out]))



model = BigramModel()
model.to(device)
# Training the model
steps = 20_000
lr = 1e-3
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for i in range(steps):
  Xb, Yb = get_batch()
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()

  if i % 500 == 0:
    evaluate_loss(eval_iters)


# Generation
# Data generation post some training
start_phrase = torch.zeros((1, block_size), dtype=torch.long, device=device)
model.generate(start_phrase, num_tokens=1_000)

DEVICE is cpu
Train loss 4.2520365715026855 Eval loss 4.2510480880737305
Train loss 2.7064623832702637 Eval loss 2.708632707595825
Train loss 2.5218536853790283 Eval loss 2.514897108078003
Train loss 2.4772019386291504 Eval loss 2.4751343727111816
Train loss 2.4547290802001953 Eval loss 2.445376396179199
Train loss 2.43082332611084 Eval loss 2.4304161071777344
Train loss 2.4216301441192627 Eval loss 2.4297704696655273
Train loss 2.4215052127838135 Eval loss 2.418318510055542
Train loss 2.4006433486938477 Eval loss 2.4246039390563965
Train loss 2.399266242980957 Eval loss 2.406872034072876
Train loss 2.3859145641326904 Eval loss 2.4093408584594727
Train loss 2.3920629024505615 Eval loss 2.4162793159484863
Train loss 2.383587598800659 Eval loss 2.391150951385498
Train loss 2.3717002868652344 Eval loss 2.4078710079193115
Train loss 2.3597774505615234 Eval loss 2.3926422595977783
Train loss 2.3829050064086914 Eval loss 2.402067184448242
Train loss 2.3539228439331055 Eval loss 2.40496134757

In [ ]:
### Multihead attention
import torch
import torch.nn as nn
import torch.nn.functional as F

with open('input.txt', mode='r', encoding='utf-8') as f:
  text = f.read()

chars = list(sorted(set(text)))
itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
vocab_size=len(chars)

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

data = torch.tensor(encode(text), dtype=torch.int64)

# Defining data split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


block_size = 8
n_embd = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"DEVICE is {device}")

def get_batch(split='train', batch_size=32):
  if split == 'train':
    data = train_data
  elif split == 'val':
    data = val_data

  ix = torch.randint(0, data.shape[0]-block_size, (batch_size,))
  list_X = [data[i: i+block_size] for i in ix] # 32 list elements with each element of size 8
  X = torch.stack(list_X, dim=0) # [32, 8]
  Y = torch.stack([data[i+1: i+block_size+1] for i in ix], dim=0) # [32, 8]
  return X.to(device), Y.to(device)

@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train', 'val']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']} Eval loss {losses['val']}")

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([AttnHead(head_size) for _ in range(num_heads)])

  def forward(self, x):
    out = [head(x) for head in self.heads]
    out = torch.cat(out, dim=-1)
    return out

class AttnHead(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.Q = nn.Linear(n_embd, head_size, bias=False) # Stores [out_features, in_features] and X @ W.T
    self.K = nn.Linear(n_embd, head_size, bias=False)
    self.V = nn.Linear(n_embd, head_size, bias=False)
    #https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, input):
    # input [B, T, n_embd]
    B,T,C = input.shape
    query = self.Q(input) # [n_embd, n_embd] broadcasted to [B, n_embd, n_embd]; [B, T, n_embd] @ [B, n_embd, n_embd]
    key = self.K(input) # [B, T, n_embd]
    value = self.V(input) # [B, T, n_embd]
    # print(f"I: {input.shape} K: {key.shape} Q: {query.shape} KT: {torch.transpose(key, -2, -1).shape}")

    wei = query @ torch.transpose(key, -2, -1) # [B, T, n_embd] @ [B, n_embd, T] -> [B, T, T]
    wei = wei * (C**-0.5) # Scaling down
    # Until this wei is initial affinities based on data

    wei = wei.masked_fill(self.tril==0, -torch.inf)
    wei = F.softmax(wei, dim=-1)
    out = wei @ value # [B,T,T] @ [B,T,n_embd] -> [B,T,n_embd]
    return out

class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
    self.position_table = nn.Embedding(block_size, n_embd)
    self.sa_heads = MultiHeadAttention(4, n_embd//4)
    self.lm_head = nn.Linear(in_features=n_embd, out_features=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    B, T = input.shape
    token_embd = self.embedding_table(input) # (B, T, embd) -> (batch, time, channels) -> channels is embedding size here
    pos_input = torch.arange(T).repeat(B,1) # [B, T] [[0,1,2...7] ... 8 batch]
    pos_embd = self.position_table(pos_input) # (B, T, embd)
    # Alternatively can do
    # pos_embd = self.position_table(torch.arange(T)) # [T, embd] (would work in next step because of broadcasting)
    x = token_embd + pos_embd
    x = self.sa_heads(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, input, num_tokens=200):
    # Initial input [1, T]
    out = []
    for _ in range(num_tokens):
      logits, _ = self(input) # [1, T, 65]
      last_T = logits[:, -1, :] # [1, 65]
      probs = F.softmax(last_T, dim=1)
      next_idx = torch.multinomial(probs, num_samples=1) # [1, 1]
      out.append(next_idx.item())
      input = torch.cat((input[:,1:], next_idx), dim=1) # [1, T-1]+[1, 1] -> [1, T]

    print(f"=========== Generated output ==========\n")
    print("".join([itos[i] for i in out]))



model = BigramModel()
model.to(device)
# Training the model
steps = 20_000
lr = 1e-3
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for i in range(steps):
  Xb, Yb = get_batch()
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()

  if i % 500 == 0:
    evaluate_loss(eval_iters)


# Generation
# Data generation post some training
start_phrase = torch.zeros((1, block_size), dtype=torch.long, device=device)
model.generate(start_phrase, num_tokens=1_000)

DEVICE is cpu
Train loss 4.186030864715576 Eval loss 4.186642169952393
Train loss 2.6658496856689453 Eval loss 2.6585376262664795
Train loss 2.504936933517456 Eval loss 2.5054872035980225
Train loss 2.4404473304748535 Eval loss 2.4309885501861572
Train loss 2.3827576637268066 Eval loss 2.4015321731567383
Train loss 2.3380937576293945 Eval loss 2.360468864440918
Train loss 2.321315050125122 Eval loss 2.326988935470581
Train loss 2.30100679397583 Eval loss 2.3101918697357178
Train loss 2.2731270790100098 Eval loss 2.2912864685058594
Train loss 2.2652587890625 Eval loss 2.2861740589141846
Train loss 2.250964403152466 Eval loss 2.269615888595581
Train loss 2.244340419769287 Eval loss 2.280442953109741
Train loss 2.245725154876709 Eval loss 2.2594552040100098
Train loss 2.2135579586029053 Eval loss 2.2469589710235596
Train loss 2.216749906539917 Eval loss 2.2524402141571045
Train loss 2.2062411308288574 Eval loss 2.2459726333618164
Train loss 2.198269844055176 Eval loss 2.2367770671844482
T

### Adding FeedForward MLP

In [ ]:
### Multihead attention
import torch
import torch.nn as nn
import torch.nn.functional as F

with open('input.txt', mode='r', encoding='utf-8') as f:
  text = f.read()

chars = list(sorted(set(text)))
itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
vocab_size=len(chars)

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

data = torch.tensor(encode(text), dtype=torch.int64)

# Defining data split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


block_size = 8
n_embd = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"DEVICE is {device}")

def get_batch(split='train', batch_size=32):
  if split == 'train':
    data = train_data
  elif split == 'val':
    data = val_data

  ix = torch.randint(0, data.shape[0]-block_size, (batch_size,))
  list_X = [data[i: i+block_size] for i in ix] # 32 list elements with each element of size 8
  X = torch.stack(list_X, dim=0) # [32, 8]
  Y = torch.stack([data[i+1: i+block_size+1] for i in ix], dim=0) # [32, 8]
  return X.to(device), Y.to(device)

@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train', 'val']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']} Eval loss {losses['val']}")

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([AttnHead(head_size) for _ in range(num_heads)])

  def forward(self, x):
    out = [head(x) for head in self.heads]
    out = torch.cat(out, dim=-1)
    return out

class AttnHead(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.Q = nn.Linear(n_embd, head_size, bias=False) # Stores [out_features, in_features] and X @ W.T
    self.K = nn.Linear(n_embd, head_size, bias=False)
    self.V = nn.Linear(n_embd, head_size, bias=False)
    #https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, input):
    # input [B, T, n_embd]
    B,T,C = input.shape
    query = self.Q(input) # [n_embd, n_embd] broadcasted to [B, n_embd, n_embd]; [B, T, n_embd] @ [B, n_embd, n_embd]
    key = self.K(input) # [B, T, n_embd]
    value = self.V(input) # [B, T, n_embd]
    # print(f"I: {input.shape} K: {key.shape} Q: {query.shape} KT: {torch.transpose(key, -2, -1).shape}")

    wei = query @ torch.transpose(key, -2, -1) # [B, T, n_embd] @ [B, n_embd, T] -> [B, T, T]
    wei = wei * (C**-0.5) # Scaling down
    # Until this wei is initial affinities based on data

    wei = wei.masked_fill(self.tril==0, -torch.inf)
    wei = F.softmax(wei, dim=-1)
    out = wei @ value # [B,T,T] @ [B,T,n_embd] -> [B,T,n_embd]
    return out

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, n_embd),
        nn.ReLU()
    )
  def forward(self, x):
    return self.net(x)

class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
    self.position_table = nn.Embedding(block_size, n_embd)
    self.sa_heads = MultiHeadAttention(4, n_embd//4)
    self.ffwd = FeedForward()
    self.lm_head = nn.Linear(in_features=n_embd, out_features=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    B, T = input.shape
    token_embd = self.embedding_table(input) # (B, T, embd) -> (batch, time, channels) -> channels is embedding size here
    pos_input = torch.arange(T).repeat(B,1) # [B, T] [[0,1,2...7] ... 8 batch]
    pos_embd = self.position_table(pos_input) # (B, T, embd)
    # Alternatively can do
    # pos_embd = self.position_table(torch.arange(T)) # [T, embd] (would work in next step because of broadcasting)
    x = token_embd + pos_embd
    x = self.sa_heads(x)
    x = self.ffwd(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, input, num_tokens=200):
    # Initial input [1, T]
    out = []
    for _ in range(num_tokens):
      logits, _ = self(input) # [1, T, 65]
      last_T = logits[:, -1, :] # [1, 65]
      probs = F.softmax(last_T, dim=1)
      next_idx = torch.multinomial(probs, num_samples=1) # [1, 1]
      out.append(next_idx.item())
      input = torch.cat((input[:,1:], next_idx), dim=1) # [1, T-1]+[1, 1] -> [1, T]

    print(f"=========== Generated output ==========\n")
    print("".join([itos[i] for i in out]))



model = BigramModel()
model.to(device)
# Training the model
steps = 20_000
lr = 1e-3
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for i in range(steps):
  Xb, Yb = get_batch()
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()

  if i % 500 == 0:
    evaluate_loss(eval_iters)


# Generation
# Data generation post some training
start_phrase = torch.zeros((1, block_size), dtype=torch.long, device=device)
model.generate(start_phrase, num_tokens=1_000)

DEVICE is cpu
Train loss 4.218749046325684 Eval loss 4.219216823577881
Train loss 2.6038970947265625 Eval loss 2.608952283859253
Train loss 2.4772250652313232 Eval loss 2.476696491241455
Train loss 2.43111252784729 Eval loss 2.428180456161499
Train loss 2.386942148208618 Eval loss 2.3980541229248047
Train loss 2.360677719116211 Eval loss 2.3656415939331055
Train loss 2.3297011852264404 Eval loss 2.3440816402435303
Train loss 2.3266663551330566 Eval loss 2.3288638591766357
Train loss 2.3028488159179688 Eval loss 2.298295021057129
Train loss 2.2783613204956055 Eval loss 2.2941057682037354
Train loss 2.2633025646209717 Eval loss 2.2943472862243652
Train loss 2.2434470653533936 Eval loss 2.2853379249572754
Train loss 2.238091468811035 Eval loss 2.266191005706787
Train loss 2.221609592437744 Eval loss 2.2472035884857178
Train loss 2.2125213146209717 Eval loss 2.2471635341644287
Train loss 2.195127010345459 Eval loss 2.2383546829223633
Train loss 2.197248697280884 Eval loss 2.232731103897094

### Creating multiple blocks

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

with open('input.txt', mode='r', encoding='utf-8') as f:
  text = f.read()

chars = list(sorted(set(text)))
itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
vocab_size=len(chars)

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

data = torch.tensor(encode(text), dtype=torch.int64)

# Defining data split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


block_size = 8
n_embd = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"DEVICE is {device}")

def get_batch(split='train', batch_size=32):
  if split == 'train':
    data = train_data
  elif split == 'val':
    data = val_data

  ix = torch.randint(0, data.shape[0]-block_size, (batch_size,))
  list_X = [data[i: i+block_size] for i in ix] # 32 list elements with each element of size 8
  X = torch.stack(list_X, dim=0) # [32, 8]
  Y = torch.stack([data[i+1: i+block_size+1] for i in ix], dim=0) # [32, 8]
  return X.to(device), Y.to(device)

@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train', 'val']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']} Eval loss {losses['val']}")

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([AttnHead(head_size) for _ in range(num_heads)])

  def forward(self, x):
    out = [head(x) for head in self.heads]
    out = torch.cat(out, dim=-1)
    return out

class AttnHead(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.Q = nn.Linear(n_embd, head_size, bias=False) # Stores [out_features, in_features] and X @ W.T
    self.K = nn.Linear(n_embd, head_size, bias=False)
    self.V = nn.Linear(n_embd, head_size, bias=False)
    #https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, input):
    # input [B, T, n_embd]
    B,T,C = input.shape
    query = self.Q(input) # [n_embd, n_embd] broadcasted to [B, n_embd, n_embd]; [B, T, n_embd] @ [B, n_embd, n_embd]
    key = self.K(input) # [B, T, n_embd]
    value = self.V(input) # [B, T, n_embd]
    # print(f"I: {input.shape} K: {key.shape} Q: {query.shape} KT: {torch.transpose(key, -2, -1).shape}")

    wei = query @ torch.transpose(key, -2, -1) # [B, T, n_embd] @ [B, n_embd, T] -> [B, T, T]
    wei = wei * (C**-0.5) # Scaling down
    # Until this wei is initial affinities based on data

    wei = wei.masked_fill(self.tril==0, -torch.inf)
    wei = F.softmax(wei, dim=-1)
    out = wei @ value # [B,T,T] @ [B,T,n_embd] -> [B,T,n_embd]
    return out

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, n_embd),
        nn.ReLU()
    )
  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.sa_heads = MultiHeadAttention(4, n_embd//4)
    self.ffwd = FeedForward()

  def forward(self, x):
    x = self.sa_heads(x)
    x = self.ffwd(x)
    return x


class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
    self.position_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(
        Block(n_embd),
        Block(n_embd),
        Block(n_embd),
    )
    self.lm_head = nn.Linear(in_features=n_embd, out_features=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    B, T = input.shape
    token_embd = self.embedding_table(input) # (B, T, embd) -> (batch, time, channels) -> channels is embedding size here
    pos_input = torch.arange(T).repeat(B,1) # [B, T] [[0,1,2...7] ... 8 batch]
    pos_embd = self.position_table(pos_input) # (B, T, embd)
    # Alternatively can do
    # pos_embd = self.position_table(torch.arange(T)) # [T, embd] (would work in next step because of broadcasting)
    x = token_embd + pos_embd
    x = self.blocks(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, input, num_tokens=200):
    # Initial input [1, T]
    out = []
    for _ in range(num_tokens):
      logits, _ = self(input) # [1, T, 65]
      last_T = logits[:, -1, :] # [1, 65]
      probs = F.softmax(last_T, dim=1)
      next_idx = torch.multinomial(probs, num_samples=1) # [1, 1]
      out.append(next_idx.item())
      input = torch.cat((input[:,1:], next_idx), dim=1) # [1, T-1]+[1, 1] -> [1, T]

    print(f"=========== Generated output ==========\n")
    print("".join([itos[i] for i in out]))



model = BigramModel()
model.to(device)
# Training the model
steps = 20_000
lr = 1e-3
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for i in range(steps):
  Xb, Yb = get_batch()
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()

  if i % 500 == 0:
    evaluate_loss(eval_iters)


# Generation
# Data generation post some training
start_phrase = torch.zeros((1, block_size), dtype=torch.long, device=device)
model.generate(start_phrase, num_tokens=1_000)

DEVICE is cpu
Train loss 4.168332099914551 Eval loss 4.168600082397461
Train loss 3.1061272621154785 Eval loss 3.099609136581421
Train loss 2.8223464488983154 Eval loss 2.821547508239746
Train loss 2.614961624145508 Eval loss 2.6036694049835205
Train loss 2.525078296661377 Eval loss 2.518231153488159
Train loss 2.485200881958008 Eval loss 2.477856159210205
Train loss 2.4527499675750732 Eval loss 2.4536900520324707
Train loss 2.3985347747802734 Eval loss 2.427279472351074
Train loss 2.3711297512054443 Eval loss 2.3844175338745117
Train loss 2.3542537689208984 Eval loss 2.365048408508301
Train loss 2.320265769958496 Eval loss 2.341607093811035
Train loss 2.3085639476776123 Eval loss 2.3311684131622314
Train loss 2.3025221824645996 Eval loss 2.3181557655334473
Train loss 2.2686681747436523 Eval loss 2.298478126525879
Train loss 2.281174659729004 Eval loss 2.287856340408325
Train loss 2.2718100547790527 Eval loss 2.2754344940185547
Train loss 2.225311756134033 Eval loss 2.2607581615448
Tra

### Note:
There is not a significant improvement by adding multiple blocks as compared to just one block.

Therefore adding:
- Skip connections (useful in deeper network)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

with open('input.txt', mode='r', encoding='utf-8') as f:
  text = f.read()

chars = list(sorted(set(text)))
itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
vocab_size=len(chars)

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

data = torch.tensor(encode(text), dtype=torch.int64)

# Defining data split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


block_size = 8
n_embd = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"DEVICE is {device}")

def get_batch(split='train', batch_size=32):
  if split == 'train':
    data = train_data
  elif split == 'val':
    data = val_data

  ix = torch.randint(0, data.shape[0]-block_size, (batch_size,))
  list_X = [data[i: i+block_size] for i in ix] # 32 list elements with each element of size 8
  X = torch.stack(list_X, dim=0) # [32, 8]
  Y = torch.stack([data[i+1: i+block_size+1] for i in ix], dim=0) # [32, 8]
  return X.to(device), Y.to(device)

@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train', 'val']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']} Eval loss {losses['val']}")

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([AttnHead(head_size) for _ in range(num_heads)])
    self.projection = nn.Linear(n_embd, n_embd)
    # Intuition for projection:
    # The projection layer is needed not just to match dimensions,
    # but to mix information across heads and align the output with the input space
    # so the residual connection works properly.
    # Think of it like:

    # Heads = specialists
    # Projection = team meeting where results are combined
    # Without projection:
    # everyone talks, nobody integrates

  def forward(self, x):
    out = [head(x) for head in self.heads]
    out = torch.cat(out, dim=-1)
    out = self.projection(out)
    return out

class AttnHead(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.Q = nn.Linear(n_embd, head_size, bias=False) # Stores [out_features, in_features] and X @ W.T
    self.K = nn.Linear(n_embd, head_size, bias=False)
    self.V = nn.Linear(n_embd, head_size, bias=False)
    #https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, input):
    # input [B, T, n_embd]
    B,T,C = input.shape
    query = self.Q(input) # [n_embd, n_embd] broadcasted to [B, n_embd, n_embd]; [B, T, n_embd] @ [B, n_embd, n_embd]
    key = self.K(input) # [B, T, n_embd]
    value = self.V(input) # [B, T, n_embd]
    # print(f"I: {input.shape} K: {key.shape} Q: {query.shape} KT: {torch.transpose(key, -2, -1).shape}")

    wei = query @ torch.transpose(key, -2, -1) # [B, T, n_embd] @ [B, n_embd, T] -> [B, T, T]
    wei = wei * (C**-0.5) # Scaling down
    # Until this wei is initial affinities based on data

    wei = wei.masked_fill(self.tril==0, -torch.inf)
    wei = F.softmax(wei, dim=-1)
    out = wei @ value # [B,T,T] @ [B,T,n_embd] -> [B,T,n_embd]
    return out

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd, n_embd) # Additional projection
    )
  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.sa_heads = MultiHeadAttention(4, n_embd//4)
    self.ffwd = FeedForward()

  def forward(self, x):
    x = x + self.sa_heads(x) # [B, T, C]
    x = x + self.ffwd(x)
    return x


class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
    self.position_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(
        Block(n_embd),
        Block(n_embd),
        Block(n_embd),
    )
    self.lm_head = nn.Linear(in_features=n_embd, out_features=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    B, T = input.shape
    token_embd = self.embedding_table(input) # (B, T, embd) -> (batch, time, channels) -> channels is embedding size here
    pos_input = torch.arange(T).repeat(B,1) # [B, T] [[0,1,2...7] ... 8 batch]
    pos_embd = self.position_table(pos_input) # (B, T, embd)
    # Alternatively can do
    # pos_embd = self.position_table(torch.arange(T)) # [T, embd] (would work in next step because of broadcasting)
    x = token_embd + pos_embd
    x = self.blocks(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, input, num_tokens=200):
    # Initial input [1, T]
    out = []
    for _ in range(num_tokens):
      logits, _ = self(input) # [1, T, 65]
      last_T = logits[:, -1, :] # [1, 65]
      probs = F.softmax(last_T, dim=1)
      next_idx = torch.multinomial(probs, num_samples=1) # [1, 1]
      out.append(next_idx.item())
      input = torch.cat((input[:,1:], next_idx), dim=1) # [1, T-1]+[1, 1] -> [1, T]

    print(f"=========== Generated output ==========\n")
    print("".join([itos[i] for i in out]))



model = BigramModel()
model.to(device)
# Training the model
steps = 20_000
lr = 1e-3
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for i in range(steps):
  Xb, Yb = get_batch()
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()

  if i % 500 == 0:
    evaluate_loss(eval_iters)


# Generation
# Data generation post some training
start_phrase = torch.zeros((1, block_size), dtype=torch.long, device=device)
model.generate(start_phrase, num_tokens=1_000)

DEVICE is cpu
Train loss 4.469487190246582 Eval loss 4.478273391723633
Train loss 2.3810458183288574 Eval loss 2.3938121795654297
Train loss 2.2597978115081787 Eval loss 2.2909042835235596
Train loss 2.1931111812591553 Eval loss 2.238743305206299
Train loss 2.1611640453338623 Eval loss 2.1940619945526123
Train loss 2.100780963897705 Eval loss 2.161651372909546
Train loss 2.0858218669891357 Eval loss 2.1486623287200928
Train loss 2.057830572128296 Eval loss 2.1443655490875244
Train loss 2.0283091068267822 Eval loss 2.1125879287719727
Train loss 2.025937080383301 Eval loss 2.098501682281494
Train loss 2.0064988136291504 Eval loss 2.1019067764282227
Train loss 1.991155743598938 Eval loss 2.0881597995758057
Train loss 1.9730638265609741 Eval loss 2.086897373199463
Train loss 1.9711288213729858 Eval loss 2.0800139904022217
Train loss 1.9612460136413574 Eval loss 2.085451364517212
Train loss 1.9496411085128784 Eval loss 2.057551622390747
Train loss 1.941506028175354 Eval loss 2.0551190376281

### Adding LayerNorm


Normalize features across each token

In (B, T, C), LayerNorm normalizes each token independently across its feature dimension (C).

LayerNorm = “normalize each token’s features”

It answers:

“Given this token’s embedding, normalize its feature distribution”

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

with open('input.txt', mode='r', encoding='utf-8') as f:
  text = f.read()

chars = list(sorted(set(text)))
itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
vocab_size=len(chars)

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

data = torch.tensor(encode(text), dtype=torch.int64)

# Defining data split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


block_size = 8
n_embd = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"DEVICE is {device}")

def get_batch(split='train', batch_size=32):
  if split == 'train':
    data = train_data
  elif split == 'val':
    data = val_data

  ix = torch.randint(0, data.shape[0]-block_size, (batch_size,))
  list_X = [data[i: i+block_size] for i in ix] # 32 list elements with each element of size 8
  X = torch.stack(list_X, dim=0) # [32, 8]
  Y = torch.stack([data[i+1: i+block_size+1] for i in ix], dim=0) # [32, 8]
  return X.to(device), Y.to(device)

@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train', 'val']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']} Eval loss {losses['val']}")

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([AttnHead(head_size) for _ in range(num_heads)])
    self.projection = nn.Linear(n_embd, n_embd)
    # Intuition for projection:
    # The projection layer is needed not just to match dimensions,
    # but to mix information across heads and align the output with the input space
    # so the residual connection works properly.
    # Think of it like:

    # Heads = specialists
    # Projection = team meeting where results are combined
    # Without projection:
    # everyone talks, nobody integrates

  def forward(self, x):
    out = [head(x) for head in self.heads]
    out = torch.cat(out, dim=-1)
    out = self.projection(out)
    return out

class AttnHead(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.Q = nn.Linear(n_embd, head_size, bias=False) # Stores [out_features, in_features] and X @ W.T
    self.K = nn.Linear(n_embd, head_size, bias=False)
    self.V = nn.Linear(n_embd, head_size, bias=False)
    #https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, input):
    # input [B, T, n_embd]
    B,T,C = input.shape
    query = self.Q(input) # [n_embd, n_embd] broadcasted to [B, n_embd, n_embd]; [B, T, n_embd] @ [B, n_embd, n_embd]
    key = self.K(input) # [B, T, n_embd]
    value = self.V(input) # [B, T, n_embd]
    # print(f"I: {input.shape} K: {key.shape} Q: {query.shape} KT: {torch.transpose(key, -2, -1).shape}")

    wei = query @ torch.transpose(key, -2, -1) # [B, T, n_embd] @ [B, n_embd, T] -> [B, T, T]
    wei = wei * (C**-0.5) # Scaling down
    # Until this wei is initial affinities based on data

    wei = wei.masked_fill(self.tril==0, -torch.inf)
    wei = F.softmax(wei, dim=-1)
    out = wei @ value # [B,T,T] @ [B,T,n_embd] -> [B,T,n_embd]
    return out

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd, n_embd) # Additional projection
    )
  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.sa_heads = MultiHeadAttention(4, n_embd//4)
    self.ffwd = FeedForward()
    self.ln = nn.LayerNorm(n_embd) # [Mean and variance taken over 32 numbers i.e. n_embd]

  def forward(self, x):
    # [Add & Norm] as in Transformers diagram
    x = x + self.sa_heads(self.ln(x)) # [B, T, C]
    x = x + self.ffwd(self.ln(x)) # [B,T,C]
    return x


class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
    self.position_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(
        Block(n_embd),
        Block(n_embd),
        Block(n_embd),
        nn.LayerNorm(n_embd)
    )
    self.lm_head = nn.Linear(in_features=n_embd, out_features=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    B, T = input.shape
    token_embd = self.embedding_table(input) # (B, T, embd) -> (batch, time, channels) -> channels is embedding size here
    pos_input = torch.arange(T, device=device).repeat(B,1) # [B, T] [[0,1,2...7] ... 8 batch]
    pos_embd = self.position_table(pos_input) # (B, T, embd)
    # Alternatively can do
    # pos_embd = self.position_table(torch.arange(T)) # [T, embd] (would work in next step because of broadcasting)
    x = token_embd + pos_embd
    x = self.blocks(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, input, num_tokens=200):
    # Initial input [1, T]
    out = []
    for _ in range(num_tokens):
      logits, _ = self(input) # [1, T, 65]
      last_T = logits[:, -1, :] # [1, 65]
      probs = F.softmax(last_T, dim=1)
      next_idx = torch.multinomial(probs, num_samples=1) # [1, 1]
      out.append(next_idx.item())
      input = torch.cat((input[:,1:], next_idx), dim=1) # [1, T-1]+[1, 1] -> [1, T]

    print(f"=========== Generated output ==========\n")
    print("".join([itos[i] for i in out]))



model = BigramModel()
model.to(device)
# Training the model
steps = 20_000
lr = 1e-3
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for i in range(steps):
  Xb, Yb = get_batch()
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()

  if i % 500 == 0:
    evaluate_loss(eval_iters)


# Generation
# Data generation post some training
start_phrase = torch.zeros((1, block_size), dtype=torch.long, device=device)
model.generate(start_phrase, num_tokens=1_000)

DEVICE is cuda
Train loss 4.407504081726074 Eval loss 4.4031572341918945
Train loss 2.4048361778259277 Eval loss 2.4112653732299805
Train loss 2.2561092376708984 Eval loss 2.2717251777648926
Train loss 2.1853673458099365 Eval loss 2.2123398780822754
Train loss 2.127856731414795 Eval loss 2.163553237915039
Train loss 2.0925018787384033 Eval loss 2.1397244930267334
Train loss 2.060798406600952 Eval loss 2.0996344089508057
Train loss 2.0406601428985596 Eval loss 2.0995566844940186
Train loss 2.009823799133301 Eval loss 2.08807110786438
Train loss 1.9863450527191162 Eval loss 2.0604236125946045
Train loss 1.9729735851287842 Eval loss 2.0767886638641357
Train loss 1.9882612228393555 Eval loss 2.058729648590088
Train loss 1.9614471197128296 Eval loss 2.0468196868896484
Train loss 1.9589471817016602 Eval loss 2.0543107986450195
Train loss 1.9425902366638184 Eval loss 2.038651704788208
Train loss 1.9252259731292725 Eval loss 2.038499116897583
Train loss 1.9184590578079224 Eval loss 2.028326749

### Trying bigger model with Dropout


Added dropout:
- after multihead and before residual connection
- after ffwd and before residual connection
- after final wei and before multiplying with v


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

with open('input.txt', mode='r', encoding='utf-8') as f:
  text = f.read()

chars = list(sorted(set(text)))
itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
vocab_size=len(chars)

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

data = torch.tensor(encode(text), dtype=torch.int64)

# Defining data split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

batch_size = 64
block_size = 256
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

steps = 5_000
eval_interval = 500
eval_iters = 200
lr = 3e-4

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"DEVICE is {device}")

def get_batch(split='train', batch_size=32):
  if split == 'train':
    data = train_data
  elif split == 'val':
    data = val_data

  ix = torch.randint(0, data.shape[0]-block_size, (batch_size,))
  list_X = [data[i: i+block_size] for i in ix] # 32 list elements with each element of size 8
  X = torch.stack(list_X, dim=0) # [32, 8]
  Y = torch.stack([data[i+1: i+block_size+1] for i in ix], dim=0) # [32, 8]
  return X.to(device), Y.to(device)

@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train', 'val']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']} Eval loss {losses['val']}")


class AttnHead(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.Q = nn.Linear(n_embd, head_size, bias=False) # Stores [out_features, in_features] and X @ W.T
    self.K = nn.Linear(n_embd, head_size, bias=False)
    self.V = nn.Linear(n_embd, head_size, bias=False)
    #https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    self.dropout = nn.Dropout(p=dropout)

  def forward(self, input):
    # input [B, T, n_embd]
    B,T,C = input.shape
    query = self.Q(input) # [n_embd, n_embd] broadcasted to [B, n_embd, n_embd]; [B, T, n_embd] @ [B, n_embd, n_embd]
    key = self.K(input) # [B, T, n_embd]
    value = self.V(input) # [B, T, n_embd]
    # print(f"I: {input.shape} K: {key.shape} Q: {query.shape} KT: {torch.transpose(key, -2, -1).shape}")

    wei = query @ torch.transpose(key, -2, -1) # [B, T, n_embd] @ [B, n_embd, T] -> [B, T, T]
    wei = wei * (C**-0.5) # Scaling down
    # Until this wei is initial affinities based on data

    wei = wei.masked_fill(self.tril==0, -torch.inf)
    wei = F.softmax(wei, dim=-1) # Wei is initial affinities based on data
    wei = self.dropout(wei) # added this for bigger model
    out = wei @ value # [B,T,T] @ [B,T,n_embd] -> [B,T,n_embd]
    return out


class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([AttnHead(head_size) for _ in range(num_heads)])
    self.projection = nn.Linear(n_embd, n_embd)
    # Intuition for projection:
    # The projection layer is needed not just to match dimensions,
    # but to mix information across heads and align the output with the input space
    # so the residual connection works properly.
    # Think of it like:

    # Heads = specialists
    # Projection = team meeting where results are combined
    # Without projection:
    # everyone talks, nobody integrates
    self.dropout = nn.Dropout(p=dropout)

  def forward(self, x):
    out = [head(x) for head in self.heads]
    out = torch.cat(out, dim=-1)
    out = self.projection(out)
    out = self.dropout(out)
    return out


class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd, n_embd), # Additional projection
        nn.Dropout(p=dropout)
    )
  def forward(self, x):
    return self.net(x)


class Block(nn.Module):
  def __init__(self, n_head, n_embd):
    super().__init__()
    self.sa_heads = MultiHeadAttention(n_head, n_embd//n_head)
    self.ffwd = FeedForward()
    self.ln = nn.LayerNorm(n_embd) # [Mean and variance taken over 32 numbers i.e. n_embd]

  def forward(self, x):
    # [Add & Norm] as in Transformers diagram
    x = x + self.sa_heads(self.ln(x)) # [B, T, C]
    x = x + self.ffwd(self.ln(x)) # [B,T,C]
    return x


class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
    self.position_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_head, n_embd) for _ in range(n_layer)])
    self.ln = nn.LayerNorm(n_embd) # Final layernorm
    self.lm_head = nn.Linear(in_features=n_embd, out_features=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    B, T = input.shape
    token_embd = self.embedding_table(input) # (B, T, embd) -> (batch, time, channels) -> channels is embedding size here
    pos_input = torch.arange(T, device=device).repeat(B,1) # [B, T] [[0,1,2...7] ... 8 batch]
    pos_embd = self.position_table(pos_input) # (B, T, embd)
    # Alternatively can do
    # pos_embd = self.position_table(torch.arange(T)) # [T, embd] (would work in next step because of broadcasting)
    x = token_embd + pos_embd
    x = self.blocks(x)
    x = self.ln(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, input, num_tokens=200):
    # Initial input [1, T]
    out = []
    for _ in range(num_tokens):
      logits, _ = self(input) # [1, T, 65]
      last_T = logits[:, -1, :] # [1, 65]
      probs = F.softmax(last_T, dim=1)
      next_idx = torch.multinomial(probs, num_samples=1) # [1, 1]
      out.append(next_idx.item())
      input = torch.cat((input[:,1:], next_idx), dim=1) # [1, T-1]+[1, 1] -> [1, T]

    print(f"=========== Generated output ==========\n")
    print("".join([itos[i] for i in out]))



model = BigramModel()
model.to(device)

# Training the model
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for i in range(steps):
  Xb, Yb = get_batch(batch_size=batch_size)
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()

  if i % eval_interval == 0:
    evaluate_loss(eval_iters)


# Generation
# Data generation post some training
start_phrase = torch.zeros((1, block_size), dtype=torch.long, device=device)
model.generate(start_phrase, num_tokens=1_000)

DEVICE is cuda
Train loss 3.669619083404541 Eval loss 3.69008731842041
Train loss 2.0489919185638428 Eval loss 2.128875970840454
Train loss 1.6297330856323242 Eval loss 1.7890948057174683
Train loss 1.4562060832977295 Eval loss 1.6609448194503784
Train loss 1.3534890413284302 Eval loss 1.589571475982666
Train loss 1.284873604774475 Eval loss 1.5448410511016846
Train loss 1.2400555610656738 Eval loss 1.5119796991348267
Train loss 1.1931486129760742 Eval loss 1.5004501342773438
Train loss 1.1501953601837158 Eval loss 1.496120810508728
Train loss 1.1202090978622437 Eval loss 1.4878935813903809
=========== Generated output ==========

SARCARonowellowinourat: a
none ounou;our lecethiroghnesta'end on't: be
noneserven some gone:
what you in your hones chutation and of such a
newly having assaticite to be lefkerewing?an, biounts off
it your enemy, and sinkin.

Why watching you tediously valouant was,
sonsily, bide us, but not to quite to due.

CAPULET:
Into plate mercy knows some.

JULIET:
I c

### Additional Exploration

In [ ]:
# Clarifying nn Embedding

e = nn.Embedding(10,3)
e.weight

Parameter containing:
tensor([[-0.9798, -0.9078, -0.5868],
        [ 2.6827,  1.7545,  0.8437],
        [ 1.0388, -0.5373, -0.6402],
        [ 0.7716, -1.7301, -1.8229],
        [-0.2643, -0.5293, -1.1075],
        [ 2.1453,  0.2320, -0.0657],
        [-2.3204,  0.8339,  0.4793],
        [ 0.2478, -0.8921,  1.1167],
        [ 0.3688, -0.5824, -1.2626],
        [ 0.4087,  1.3733,  0.2149]], requires_grad=True)

In [ ]:
t = torch.tensor([[1, 0, 5, 9], [2, 6, 8, 7]])
e(t), e(t).shape

(tensor([[[ 2.6827,  1.7545,  0.8437],
          [-0.9798, -0.9078, -0.5868],
          [ 2.1453,  0.2320, -0.0657],
          [ 0.4087,  1.3733,  0.2149]],
 
         [[ 1.0388, -0.5373, -0.6402],
          [-2.3204,  0.8339,  0.4793],
          [ 0.3688, -0.5824, -1.2626],
          [ 0.2478, -0.8921,  1.1167]]], grad_fn=<EmbeddingBackward0>),
 torch.Size([2, 4, 3]))

In [ ]:
t2 = torch.tensor([])